# Powder Diffraction: What Actually Sets The Peak Heights

Ask why the low-angle peaks of a powder pattern are the big ones and the obvious answer — that the
low-index reflections have the largest structure factors — is wrong. For face-centred nickel every
allowed reflection has the *same* geometric structure factor, exactly 4, as tutorial 04 computed. And
the multiplicity works the other way: $\{331\}$ has 24 equivalent planes against $\{111\}$'s 8, so on
structure alone the pattern would *rise* with angle.

What shapes a powder pattern is a factor that is not about the crystal at all:

$$I_{hkl} \;\propto\; \underbrace{m_{hkl}}_{\text{multiplicity}}\;
\underbrace{|F_{hkl}|^{2}}_{\text{structure}}\;
\underbrace{\frac{1+\cos^{2}2\theta}{\sin^{2}\theta\,\cos\theta}}_{\text{Lorentz–polarization}}$$

The third factor is pure measurement geometry — how many randomly oriented crystallites can satisfy
Bragg at a given angle, how long the resulting ring is, and how a free electron radiates. It diverges
as $\theta \to 0$, has a minimum somewhere past $90^\circ$, and rises again in back-reflection.

This tutorial takes the product apart. The pattern it computes has its *strongest* peak at
$144.7^\circ$, which no real nickel pattern does — and running down the reasons for that discrepancy
is a more useful exercise than reproducing a textbook figure, because every one of them is a term the
kinematic model leaves out.

## Learning goals

1. What are the factors in a powder intensity, and which one explains the shape of the envelope?
2. Where does the Lorentz–polarization factor come from, and why does it diverge at one end and
   recover at the other?
3. How does changing the anode change the pattern, quantitatively?
4. What is the $K\alpha_2$ shoulder, why does it grow with angle, and how far does it move?
5. When is peak *height* not a measure of intensity, and why is this model's back-reflection peak too
   strong?

## 0. Setup

Nickel and zirconium from the pinned fixtures, and copper $K\alpha$ radiation.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings("ignore", message="Issues encountered while parsing CIF")
warnings.filterwarnings("ignore", message="No _symmetry_equiv_pos_as_xyz")

from pytex import (
    FrameDomain,
    RadiationSpec,
    ReferenceFrame,
    generate_powder_reflections,
    generate_xrd_pattern,
    get_phase_fixture,
    list_phase_fixtures,
    plot_xrd_pattern,
)
from pytex.diffraction.physics import lorentz_polarization_factor

np.set_printoptions(precision=4, suppress=True)

CRYSTAL = ReferenceFrame("crystal", FrameDomain.CRYSTAL, ("a", "b", "c"))
NICKEL = get_phase_fixture("ni_fcc").load_phase(crystal_frame=CRYSTAL)
ZIRCONIUM = get_phase_fixture("zr_hcp").load_phase(crystal_frame=CRYSTAL)

CU = RadiationSpec.cu_ka()
MO = RadiationSpec.mo_ka()
CO = RadiationSpec.co_ka()


def label(indices):
    return str(tuple(int(value) for value in np.asarray(indices).ravel()))


print(f"{'anode':<10} {'K-alpha1 (A)':>14} {'K-alpha2 (A)':>14} {'kind':>8}")
for radiation in (CU, MO, CO):
    print(f"{str(radiation.anode):<10} {radiation.wavelength_angstrom:>14.5f} "
          f"{str(radiation.kalpha2_wavelength_angstrom):>14} {radiation.kind:>8}")

## 1. Bragg's law fixes the positions, and nothing else

$$\lambda = 2 d_{hkl}\sin\theta
\qquad\Longrightarrow\qquad
2\theta_{hkl} = 2\arcsin\!\frac{\lambda}{2 d_{hkl}} .$$

Peak *position* is geometry and wavelength, full stop: no atoms, no intensities, no instrument. The
condition $\lambda \le 2d$ also sets a hard limit — a reflection with $d < \lambda/2$ cannot be
recorded at any angle, which is why the accessible reflection list depends on the anode.

In [ ]:
REFLECTIONS = generate_powder_reflections(
    NICKEL, radiation=CU, two_theta_range_deg=(20.0, 145.0), max_index=4
)
print(f"nickel with Cu K-alpha1, lambda = {CU.wavelength_angstrom:.5f} A\n")
print(f"{'hkl':<10} {'d (A)':>8} {'2 theta (deg)':>14} {'from Bragg':>12}")
for reflection in REFLECTIONS:
    bragg = 2.0 * np.degrees(
        np.arcsin(CU.wavelength_angstrom / (2.0 * reflection.d_spacing_angstrom))
    )
    print(f"{label(reflection.miller_indices):<10} {reflection.d_spacing_angstrom:>8.4f} "
          f"{reflection.two_theta_deg:>14.4f} {bragg:>12.4f}")
    assert abs(reflection.two_theta_deg - bragg) < 1e-9

limit = CU.wavelength_angstrom / 2.0
print(f"\nno reflection with d < lambda/2 = {limit:.4f} A can be recorded at all;")
print(f"the finest spacing in this list is {min(r.d_spacing_angstrom for r in REFLECTIONS):.4f} A")

## 2. The intensity is a product, and it factorizes exactly

$$I_{hkl} = m_{hkl}\,|F_{hkl}|^{2}\, L(\theta)$$

with $m$ the number of symmetry-equivalent planes contributing at that $2\theta$, $|F|^2$ the
structure factor from tutorial 04, and $L$ the Lorentz–polarization factor. PyTex reports all three
alongside the intensity, which makes the factorization checkable rather than a claim.

In [ ]:
print(f"{'hkl':<10} {'2theta':>8} {'m':>4} {'|F|':>9} {'|F|^2':>11} {'LP':>9} "
      f"{'m |F|^2 LP':>13} {'reported I':>12}")
products = []
for reflection in REFLECTIONS:
    modulus = reflection.structure_factor_amplitude
    product = reflection.multiplicity * modulus**2 * reflection.lorentz_polarization_factor
    products.append(product)
    print(f"{label(reflection.miller_indices):<10} {reflection.two_theta_deg:>8.3f} "
          f"{reflection.multiplicity:>4} {modulus:>9.2f} {modulus**2:>11.1f} "
          f"{reflection.lorentz_polarization_factor:>9.4f} {product:>13.4g} "
          f"{reflection.intensity:>12.4g}")

products = np.asarray(products)
reported = np.asarray([reflection.intensity for reflection in REFLECTIONS])
scale = reported[0] / products[0]
print(f"\nthe reported intensities are the product times a single constant {scale:.6g};")
print(f"largest relative deviation over the whole list: "
      f"{np.abs(products * scale / reported - 1.0).max():.3e}")
assert np.abs(products * scale / reported - 1.0).max() < 1e-9

The factorization is exact to machine precision, so the three columns are genuinely independent
explanations and can be examined one at a time.

Note the $|F|$ column: it is *constant* for nickel, because every allowed fcc reflection has the same
four atoms in phase and the atomic scattering factor in this model does not fall with angle. Whatever
shapes the pattern, it is not the structure factor.

## 3. The Lorentz–polarization factor, and why it diverges

Two unrelated effects, conventionally multiplied together.

**The Lorentz part** is geometric. In a powder, the crystallites that satisfy the Bragg condition for
one reflection lie on a cone, and the fraction of randomly oriented crystallites on that cone — plus
the length of the diffraction ring the detector intercepts — both depend on $\theta$. Combining them
gives $1/(\sin^2\theta\cos\theta)$.

**The polarization part** is electromagnetic. An unpolarized incident beam scatters with the Thomson
factor $(1 + \cos^{2}2\theta)/2$: a free electron radiates nothing along its acceleration, so
scattering at $2\theta = 90^\circ$ is half as strong as forward or backward.

$$L(\theta) = \frac{1+\cos^{2}2\theta}{\sin^{2}\theta\,\cos\theta}$$

As $\theta \to 0$ the $\sin^{2}\theta$ in the denominator sends this to infinity. The divergence is
not physical — no real diffractometer measures at $2\theta = 0$, where the direct beam is — but it
dominates the shape of every powder pattern's envelope.

In [ ]:
two_theta = np.linspace(5.0, 160.0, 2000)
theta = np.radians(two_theta / 2.0)
lorentz = 1.0 / (np.sin(theta) ** 2 * np.cos(theta))
polarization = (1.0 + np.cos(np.radians(two_theta)) ** 2) / 2.0
combined = np.asarray([lorentz_polarization_factor(value) for value in np.radians(two_theta)])

print(f"{'2 theta':>9} {'Lorentz':>12} {'polarization':>13} {'library LP':>12}")
for value in (10.0, 30.0, 60.0, 90.0, 120.0, 150.0):
    index = int(np.argmin(np.abs(two_theta - value)))
    print(f"{value:>9.0f} {lorentz[index]:>12.4f} {polarization[index]:>13.4f} "
          f"{combined[index]:>12.4f}")
ratio = (combined[int(np.argmin(np.abs(two_theta - 10.0)))]
         / combined[int(np.argmin(np.abs(two_theta - 90.0)))])
print(f"\nLP falls by a factor of {ratio:.0f} between 10 and 90 degrees, then rises again as the")
print("polarization factor recovers towards back-reflection.")
minimum = two_theta[int(np.argmin(combined))]
print(f"LP is smallest at 2 theta = {minimum:.1f} deg, not at 90: the two factors have different")
print("shapes and their product has its own minimum. Over the 44 to 145 degree range this")
print("pattern actually covers, LP spans a factor of only")
window = combined[(two_theta > 44.0) & (two_theta < 145.0)]
print(f"  {window.max() / window.min():.1f} -- large enough to reorder the peaks, far short of the")
print("  divergence, which lies below any accessible angle.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.2))
axes[0].semilogy(two_theta, lorentz, label=r"Lorentz $1/(\sin^2\theta\cos\theta)$")
axes[0].semilogy(two_theta, polarization, label=r"polarization $(1+\cos^2 2\theta)/2$")
axes[0].semilogy(two_theta, combined, "k", lw=1.8, label="product (LP)")
axes[0].set_xlabel(r"$2\theta$ (deg)"), axes[0].set_ylabel("factor")
axes[0].set_title("the two effects, and their product")
axes[0].legend(fontsize=8)

# The pattern envelope, with and without LP.
positions = np.asarray([reflection.two_theta_deg for reflection in REFLECTIONS])
with_lp = np.asarray([reflection.intensity for reflection in REFLECTIONS])
without_lp = np.asarray([
    reflection.multiplicity * reflection.structure_factor_amplitude**2
    for reflection in REFLECTIONS
])
axes[1].stem(positions, with_lp / with_lp.max(), linefmt="C0-", markerfmt="C0o",
             basefmt=" ", label="with LP (as measured)")
axes[1].stem(positions + 0.9, without_lp / without_lp.max(), linefmt="C3-", markerfmt="C3s",
             basefmt=" ", label=r"$m\,|F|^2$ only")
for position, reflection in zip(positions, REFLECTIONS):
    axes[1].annotate(label(reflection.miller_indices).replace(" ", ""),
                     (position, 1.03), fontsize=6.5, rotation=90, ha="center")
axes[1].set_ylim(0, 1.25)
axes[1].set_xlabel(r"$2\theta$ (deg)"), axes[1].set_ylabel("relative intensity")
axes[1].set_title("nickel: LP is what makes the envelope fall")
axes[1].legend(fontsize=8)
fig.tight_layout()

> **The awe note.** The red stems are what a powder pattern would look like if only the crystal
> mattered, and they *rise* with angle because the higher-index families have more members. The blue
> stems — the pattern as this model computes it — fall and then rise again, following
> $1/(\sin^2\theta\cos\theta)$ times the Thomson factor. Neither the rise nor the fall is a property
> of nickel. The most conspicuous feature of every powder pattern ever recorded belongs to the
> *measurement geometry*, and quantitative analysis has to divide it out before any number means
> something about the structure.
>
> Notice what that implies about the strongest peak in the blue set: it is $(133)$ at $144.7^\circ$,
> not $(111)$. A real nickel pattern does not look like that, and the discrepancy is informative
> rather than embarrassing. Three terms this model omits all suppress high angle, and all three grow
> with $\sin\theta/\lambda$: the atomic form factor's fall-off (the default intensity model uses $Z$
> instead of $f(s)$), the Debye–Waller factor (these fixtures carry no $B_{\text{iso}}$), and
> absorption. Section 8 lists them as limits; here they are visible as a peak in the wrong place,
> which is the more useful way to meet them.

## 4. Changing the anode changes every position

The pattern is not a property of the sample alone. Switch from Cu to Mo and every peak moves, because
$\sin\theta \propto \lambda$. Molybdenum's shorter wavelength compresses the whole pattern to low
angle — and admits reflections that copper cannot reach at all.

In [ ]:
print(f"{'hkl':<10} {'d (A)':>8}" + "".join(f"{str(r.anode) + ' 2theta':>14}" for r in (CU, MO, CO)))
by_anode = {}
for radiation in (CU, MO, CO):
    by_anode[radiation.anode] = {
        label(reflection.miller_indices): reflection.two_theta_deg
        for reflection in generate_powder_reflections(
            NICKEL, radiation=radiation, two_theta_range_deg=(2.0, 160.0), max_index=4
        )
    }
keys = list(by_anode[CU.anode])
for key in keys:
    spacing = CU.wavelength_angstrom / (
        2.0 * np.sin(np.radians(by_anode[CU.anode][key] / 2.0))
    )
    row = "".join(f"{by_anode[r.anode].get(key, float('nan')):>14.3f}" for r in (CU, MO, CO))
    print(f"{key:<10} {spacing:>8.4f}{row}")

print(f"\nreflections reachable below 160 deg:")
for radiation in (CU, MO, CO):
    print(f"  {str(radiation.anode):<4} lambda = {radiation.wavelength_angstrom:.5f} A -> "
          f"{len(by_anode[radiation.anode])} reflections")
print("\nMo reaches more reflections because lambda/2 is smaller, so finer spacings qualify.")

## 5. The $K\alpha$ doublet: a shoulder that walks

A laboratory X-ray tube emits two closely spaced lines, $K\alpha_1$ and $K\alpha_2$, with an intensity
ratio of about 2:1. Every peak is therefore a pair, and the separation is not constant. Differentiating
Bragg's law at fixed $d$,

$$\Delta(2\theta) = \frac{2\,\Delta\lambda}{\lambda}\tan\theta,$$

so the splitting grows as $\tan\theta$ — invisible at low angle and obvious at high angle. That is why
a high-angle peak in a real pattern looks asymmetric or doubled while the first peak looks clean, and
why mistaking it for a second phase is a classic beginner's error.

In [ ]:
doublet = RadiationSpec.cu_ka_doublet()
delta_lambda = doublet.kalpha2_wavelength_angstrom - doublet.wavelength_angstrom
print(f"Cu K-alpha1 {doublet.wavelength_angstrom:.5f} A, "
      f"K-alpha2 {doublet.kalpha2_wavelength_angstrom:.5f} A, "
      f"difference {delta_lambda:.5f} A")
print(f"intensity ratio alpha2/alpha1 = {doublet.kalpha2_relative_intensity:.2f}\n")

print(f"{'hkl':<10} {'2theta a1':>11} {'2theta a2':>11} {'measured split':>15} "
      f"{'2 dlambda tan(theta) / lambda':>30}")
for reflection in REFLECTIONS:
    theta = np.radians(reflection.two_theta_deg / 2.0)
    second = 2.0 * np.degrees(
        np.arcsin(doublet.kalpha2_wavelength_angstrom / (2.0 * reflection.d_spacing_angstrom))
    )
    measured = second - reflection.two_theta_deg
    predicted = np.degrees(2.0 * delta_lambda / doublet.wavelength_angstrom * np.tan(theta))
    print(f"{label(reflection.miller_indices):<10} {reflection.two_theta_deg:>11.4f} "
          f"{second:>11.4f} {measured:>15.4f} {predicted:>30.4f}")
print("\nThe closed form tracks the exact splitting to a few thousandths of a degree, and the")
print("splitting grows by an order of magnitude across the pattern.")

In [ ]:
pattern_single = generate_xrd_pattern(
    NICKEL, radiation=CU, two_theta_range_deg=(40.0, 150.0), resolution_deg=0.01,
    broadening_fwhm_deg=0.10,
)
pattern_doublet = generate_xrd_pattern(
    NICKEL, radiation=doublet, two_theta_range_deg=(40.0, 150.0), resolution_deg=0.01,
    broadening_fwhm_deg=0.10,
)

fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.2))
plot_xrd_pattern(pattern_doublet, ax=axes[0])
axes[0].set_title("nickel, Cu K-alpha doublet, FWHM 0.10 deg")

for axis_limits, ax in (((42.0, 46.0), axes[1]),):
    ax.plot(pattern_single.two_theta_grid_deg, pattern_single.intensity_grid, label="K-alpha1 only")
    ax.plot(pattern_doublet.two_theta_grid_deg, pattern_doublet.intensity_grid, label="doublet")
    ax.set_xlim(*axis_limits)
    ax.set_xlabel(r"$2\theta$ (deg)"), ax.set_ylabel("intensity")
    ax.set_title(r"the (111) peak: the shoulder is barely there")
    ax.legend(fontsize=8)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 4.0))
ax.plot(pattern_single.two_theta_grid_deg, pattern_single.intensity_grid, label="K-alpha1 only")
ax.plot(pattern_doublet.two_theta_grid_deg, pattern_doublet.intensity_grid, label="doublet")
ax.set_xlim(140.0, 150.0)
ax.set_xlabel(r"$2\theta$ (deg)"), ax.set_ylabel("intensity")
ax.set_title("the same two curves at high angle: now the doublet is resolved")
ax.legend(fontsize=8)
fig.tight_layout()

Same two curves, two windows. At the $(111)$ peak the $K\alpha_2$ contribution is a slight asymmetry;
at the highest-angle peak it is a separate line. Nothing about the sample changed between the two
panels.

## 6. Profiles: what the peak shape is for

The reflection list is a set of delta functions. A measured pattern has width, from the instrument
(slits, monochromator, detector) and from the sample (crystallite size, strain). PyTex applies a
profile with a constant FWHM by default, or the Caglioti angular dependence

$$\text{FWHM}^{2}(\theta) = U\tan^{2}\theta + V\tan\theta + W,$$

which is the standard Rietveld parametrization of instrumental broadening.

In [ ]:
configurations = (
    ("Gaussian, FWHM 0.05", {"broadening_fwhm_deg": 0.05, "profile": "gaussian"}),
    ("Gaussian, FWHM 0.30", {"broadening_fwhm_deg": 0.30, "profile": "gaussian"}),
    ("pseudo-Voigt, eta 0.5", {"broadening_fwhm_deg": 0.15, "profile": "pseudo_voigt"}),
    ("Caglioti U V W", {"caglioti_uvw": (0.02, -0.008, 0.006)}),
)
fig, ax = plt.subplots(figsize=(8.6, 4.4))
for name, options in configurations:
    pattern = generate_xrd_pattern(
        NICKEL, radiation=CU, two_theta_range_deg=(40.0, 105.0), resolution_deg=0.01, **options
    )
    grid = np.asarray(pattern.intensity_grid)
    ax.plot(pattern.two_theta_grid_deg, grid / grid.max(), lw=1.1, label=name)
ax.set_xlabel(r"$2\theta$ (deg)"), ax.set_ylabel("normalized intensity")
ax.set_title("the same reflection list under four profile models")
ax.legend(fontsize=8)
fig.tight_layout()

for name, options in configurations:
    pattern = generate_xrd_pattern(
        NICKEL, radiation=CU, two_theta_range_deg=(40.0, 105.0), resolution_deg=0.01, **options
    )
    grid = np.asarray(pattern.intensity_grid)
    integral = float(np.trapezoid(grid, np.asarray(pattern.two_theta_grid_deg)))
    print(f"{name:<24} peak max {grid.max():>12.4g}   integrated {integral:>12.4g}")

Read the two columns together. `generate_xrd_pattern` normalizes to unit peak height, so the maximum
is 1 by construction and it is the *integral* that moves — by a factor of six between the narrowest
and widest profile. Their ratio is the effective width, and that is the point: **height and area are
different measurements, and only one of them carries the structure.** A profile choice changes the
relation between them by a factor of six while changing nothing about the crystal, so an analysis that
ranks peaks by height is measuring the broadening as much as the material. Section 7 puts a number on
the damage.

## 7. Failure modes, deliberately triggered

**(a) Reading peak height as intensity when the width varies with angle.** With a Caglioti profile the
width grows with $\tan\theta$, so the high-angle peaks are spread thinner and their heights fall for a
reason that has nothing to do with the structure.

In [ ]:
narrow = generate_xrd_pattern(NICKEL, radiation=CU, two_theta_range_deg=(40.0, 150.0),
                              resolution_deg=0.01, broadening_fwhm_deg=0.08)
widening = generate_xrd_pattern(NICKEL, radiation=CU, two_theta_range_deg=(40.0, 150.0),
                                resolution_deg=0.01, caglioti_uvw=(0.05, -0.01, 0.004))

def peak_heights(pattern, positions, window=0.8):
    grid = np.asarray(pattern.two_theta_grid_deg)
    values = np.asarray(pattern.intensity_grid)
    heights = []
    for position in positions:
        mask = np.abs(grid - position) < window
        heights.append(float(values[mask].max()) if np.any(mask) else np.nan)
    return np.asarray(heights)

visible = [r for r in REFLECTIONS if 40.0 < r.two_theta_deg < 150.0]
positions = np.asarray([r.two_theta_deg for r in visible])
truth = np.asarray([r.intensity for r in visible])
constant = peak_heights(narrow, positions)
growing = peak_heights(widening, positions)

print(f"{'hkl':<10} {'2theta':>8} {'true I (rel)':>13} {'height, constant FWHM':>23} "
      f"{'height, Caglioti':>18}")
for reflection, position, value, flat, wide in zip(visible, positions, truth, constant, growing):
    print(f"{label(reflection.miller_indices):<10} {position:>8.2f} {value / truth[0]:>13.4f} "
          f"{flat / constant[0]:>23.4f} {wide / growing[0]:>18.4f}")
print("\nUnder a constant width the heights track the true intensities. Under a realistic")
print("angle-dependent width the high-angle peaks are systematically under-reported --")
print("here by more than a factor of two on the last reflection.")
assert growing[-1] / growing[0] < constant[-1] / constant[0]

**(b) Forgetting the Lorentz–polarization factor when comparing with a calculation.** The correction
spans two orders of magnitude across a pattern, so an uncorrected comparison misranks the peaks
entirely.

In [ ]:
uncorrected = np.asarray([r.multiplicity * r.structure_factor_amplitude**2 for r in visible])
corrected = np.asarray([r.intensity for r in visible])
order_uncorrected = np.argsort(-uncorrected)
order_corrected = np.argsort(-corrected)
print("strongest first, ranked without LP:")
print("  " + "  ".join(label(visible[i].miller_indices).replace(" ", "") for i in order_uncorrected))
print("strongest first, ranked with LP (as measured):")
print("  " + "  ".join(label(visible[i].miller_indices).replace(" ", "") for i in order_corrected))
print(f"\nLP spans a factor of "
      f"{max(r.lorentz_polarization_factor for r in visible) / min(r.lorentz_polarization_factor for r in visible):.0f} "
      f"across this pattern.")
assert not np.array_equal(order_uncorrected, order_corrected)

**(c) Comparing patterns taken with different anodes by $2\theta$.** Peak positions are not a
fingerprint of the material; $d$-spacings are. Two patterns of the same nickel, indexed by angle,
share nothing.

In [ ]:
copper = {label(r.miller_indices): r.two_theta_deg for r in
          generate_powder_reflections(NICKEL, radiation=CU, two_theta_range_deg=(5.0, 155.0),
                                      max_index=3)}
molybdenum = {label(r.miller_indices): r.two_theta_deg for r in
              generate_powder_reflections(NICKEL, radiation=MO, two_theta_range_deg=(5.0, 155.0),
                                          max_index=3)}
shared = [key for key in copper if key in molybdenum]
print(f"{'hkl':<10} {'Cu 2theta':>11} {'Mo 2theta':>11} {'ratio of sin(theta)':>21}")
for key in shared:
    ratio = np.sin(np.radians(molybdenum[key] / 2.0)) / np.sin(np.radians(copper[key] / 2.0))
    print(f"{key:<10} {copper[key]:>11.3f} {molybdenum[key]:>11.3f} {ratio:>21.5f}")
print(f"\nEvery ratio is lambda(Mo) / lambda(Cu) = "
      f"{MO.wavelength_angstrom / CU.wavelength_angstrom:.5f}, exactly, because sin(theta)")
print("is proportional to lambda at fixed d. Index by d, never by 2 theta.")
ratios = [np.sin(np.radians(molybdenum[k] / 2.0)) / np.sin(np.radians(copper[k] / 2.0))
          for k in shared]
assert np.allclose(ratios, MO.wavelength_angstrom / CU.wavelength_angstrom, atol=1e-9)

## 8. What this implementation does not do

- **The default intensity model uses atomic numbers, not tabulated form factors.**
  `intensity_model="xray_atomic_number"` replaces $f(s)$ by $Z$, which is why the $|F|$ column of
  section 2 is flat and why the high-angle peaks are too strong. `"xray_tabulated"` uses the fitted
  form factors and restores the fall-off with $s$; the atomic-number model is the cheap default and is
  stated as such rather than hidden.
- **No Debye–Waller factor unless the phase carries one.** The pinned fixtures have `b_iso = None`,
  so these are zero-temperature intensities. Real high-angle peaks are weaker than computed here.
- **No absorption, no extinction, no surface roughness.** The Lorentz–polarization factor is the only
  geometric correction applied. A quantitative Rietveld refinement needs more.
- **Profiles are symmetric.** Axial divergence makes real low-angle peaks asymmetric, and that is not
  modelled; the Caglioti form covers the width, not the shape.
- **One phase at a time.** There is no multi-phase mixture model here, and no scale-factor
  refinement.

## 9. What to take away

- **Positions are geometry; intensities are a product.** $I = m|F|^2 L$, verified to factorize
  exactly, so each factor can be examined and corrected on its own.
- **The envelope is the Lorentz–polarization factor.** For nickel $m|F|^2$ actually *rises* with
  angle; the measured envelope falls because $L$ does, and $L$ is measurement geometry. It also
  *recovers* past its minimum, which is why this kinematic model puts its strongest peak at
  $144.7^\circ$ — and why the terms of section 8 are not optional for a real comparison.
- **Area, not height.** At fixed peak height the integrated area varies by a factor of six across
  reasonable profiles, and an angle-dependent width biases heights systematically — by more than a
  factor of seven on the highest-angle peak here.
- **Index by $d$, never by $2\theta$.** Change the anode and every $\sin\theta$ scales by
  $\lambda_2/\lambda_1$ — exactly, as verified above.
- **The $K\alpha_2$ shoulder grows as $\tan\theta$.** Invisible on the first peak, resolved on the
  last, and not a second phase.

### Further reading

- Tutorial 04, *Phases, lattices, space groups and CIF* — where $|F_{hkl}|$ and the selection rules
  come from.
- Tutorial 03, *Crystal symmetry* — the orbit counting that gives the multiplicity $m$.
- Tutorial 12, *SAED workflows* — the same structure factors in electron diffraction, where the
  Lorentz–polarization factor does not apply and the geometry is a projection instead.
- Tutorial 25, *Pole-figure arithmetic* — texture, which is what breaks the random-orientation
  assumption the Lorentz factor is derived from.
- `docs/site/theory/powder_xrd_and_saed.md` — the intensity model, the profile functions, and the
  Caglioti parametrization.
- `docs/site/theory/preferred_orientation_in_powder_intensities.md` — the March–Dollase
  correction for textured powders.
- B. D. Cullity and S. R. Stock, *Elements of X-ray Diffraction*, 3rd ed. (Prentice Hall, 2001),
  Ch. 4 and Appendix 6 — the Lorentz–polarization factor derived from the powder geometry.
- G. Caglioti, A. Paoletti and F. P. Ricci, *Nucl. Instrum.* **3** (1958) 223 — the FWHM
  parametrization.